# RSNA Knee — conversion T1 coronal/axial → NPZ (crop 130 mm)

**Shard 0/2** (CPU, internet off). **224×224** uint8.

Complement v20b de `convert_crop130_part*` + `convert_crop130_t1_part*` : **1 T1 coronal** et **1 T1 axial** par étude s'ils existent, hors des 3 picks fluid et hors du T1 sagittal déjà extrait. Dans ce CSV `Fluid_Sensitive == Fat_Suppression` : pas de fluide sans FS, donc AX_T1 remplace le slot vide `SAG_FLUID_NOFS`.

**Même recette** que crop130 v1 :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 % si spacing absent
2. bande profondeur **20–80 %**
3. resample **24** coupes, resize **224×224**, uint8, percentiles 1–99

Sortie dans `/kaggle/working/` :
- `volumes_crop130_proto_part0.zip` (`{SeriesInstanceUID}.npz` → `data`)
- `manifest_proto_part0.csv`
- `RECIPE.json`

A fusionner ensuite dans `data/crop130/` (zips **en plus**). Le submit DICOM n'a pas besoin de ce zip : `preprocess_dicom` relira les slots.

In [ ]:
SHARD_ID = 0
N_SHARDS = 2

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 224
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        # debug local : CSV seulement, pas de DICOM
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
# Kaggle : train_series/<StudyUID>/<SeriesUID>/*.dcm
# (parfois a la racine COMP_DIR, parfois un cran plus bas)
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:

series_df = pd.read_csv(TRAIN_SERIES_CSV)
print("series", len(series_df), "| studies", series_df.StudyInstanceUID.nunique())

def pick_fluid_sids(g: pd.DataFrame) -> set[str]:
    """Meme 3 picks que crop130 (1 fluid-prioritaire / plan)."""
    used: set[str] = set()
    for plane in PLANES:
        gp = g[g.Anatomical_Plane == plane].sort_values("SeriesInstanceUID")
        if len(gp) == 0:
            continue
        fluid = gp[gp.Fluid_Sensitive == 1] if "Fluid_Sensitive" in gp.columns else gp.iloc[0:0]
        if len(fluid) == 0 and "Fat_Suppression" in gp.columns:
            fluid = gp[gp.Fat_Suppression == 1]
        row = fluid.iloc[0] if len(fluid) else gp.iloc[0]
        used.add(str(row.SeriesInstanceUID))
    return used

def pick_t1_extra(g: pd.DataFrame, exclude: set[str]):
    """Meme 1 non-fluide que convert_crop130_t1 (sagittal d'abord)."""
    for plane in PLANES:
        gp = g[g.Anatomical_Plane == plane].sort_values("SeriesInstanceUID")
        if "Fluid_Sensitive" not in gp.columns:
            continue
        nf = gp[gp.Fluid_Sensitive == 0]
        for _, row in nf.iterrows():
            if str(row.SeriesInstanceUID) not in exclude:
                return row
    return None

def pick_first(g: pd.DataFrame, plane: str, fluid: int, exclude: set[str]):
    gp = g[g.Anatomical_Plane == plane].sort_values("SeriesInstanceUID")
    if "Fluid_Sensitive" not in gp.columns:
        return None
    gp = gp[gp.Fluid_Sensitive == fluid]
    for _, row in gp.iterrows():
        if str(row.SeriesInstanceUID) not in exclude:
            return row
    return None

jobs = []
studies = sorted(series_df.StudyInstanceUID.unique())
my_studies = [u for i, u in enumerate(studies) if i % N_SHARDS == SHARD_ID]
print(f"etudes ce shard : {len(my_studies)} / {len(studies)}")

n_skip = 0
n_cor = n_ax = 0
for uid, g in series_df[series_df.StudyInstanceUID.isin(set(my_studies))].groupby("StudyInstanceUID"):
    used = pick_fluid_sids(g)
    extra = pick_t1_extra(g, used)
    if extra is not None:
        used.add(str(extra.SeriesInstanceUID))
    n_before = len(jobs)
    for slot, plane in (("COR_T1", "Coronal"), ("AX_T1", "Axial")):
        row = pick_first(g, plane, 0, used)
        if row is None:
            continue
        sid = str(row.SeriesInstanceUID)
        used.add(sid)
        jobs.append({
            "study": uid,
            "series": sid,
            "plane": row.Anatomical_Plane,
            "fluid": int(row.Fluid_Sensitive) if "Fluid_Sensitive" in row.index else 0,
            "slot": slot,
        })
        if slot == "COR_T1":
            n_cor += 1
        else:
            n_ax += 1
    if len(jobs) == n_before:
        n_skip += 1

print("COR_T1", n_cor, "| AX_T1", n_ax, "| etudes sans extra :", n_skip)
print("series proto a convertir :", len(jobs))
assert jobs, "aucune serie proto selectionnee"


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    # pad/crop spatial identique : si shapes divergent (rare), resize au mode
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w)

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    band = vol[a:b + 1]
    idx = np.linspace(0, band.shape[0] - 1, N_SLICES).round().astype(int)
    band = band[idx]

    out = np.empty((N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for i in range(N_SLICES):
        sl = cv2.resize(band[i], (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        out[i] = np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    # fallback : parfois series a la racine de train_series
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_proto_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_proto_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "note": "1 COR_T1 + 1 AX_T1 si dispo, hors 3 picks fluid et hors T1 sag extra. Crop 130 mm (PixelSpacing) sinon centre 70 %. Bande profondeur 20-80 %, linspace 24, resize 224, uint8.",
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

# ZipFile n'est pas thread-safe en ecriture : workers convertissent, le
# thread principal ecrit le zip.
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

# Sanity : recharger 3 NPZ, shapes, moments
import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    means = []
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        print(name, vol.shape, vol.dtype, "min", int(vol.min()), "max", int(vol.max()),
              "mean", float(vol.mean()), "std", float(vol.std()))
        assert vol.shape == (N_SLICES, IMG_SIZE, IMG_SIZE), vol.shape
        means.append(float(vol.mean()))
print("mean uint8 (3 samples)", means)
print(f"attendu : shape ({N_SLICES}, {IMG_SIZE}, {IMG_SIZE}) uint8")
print("cible historique NPZ communautaires : mean~42 std~49.5 (crop 130 mm => moments differents, c'est normal)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
